In [1]:
# --- iPython Config --- #
from IPython import get_ipython
if 'IPython.extensions.autoreload' not in get_ipython().extension_manager.loaded:
    get_ipython().run_line_magic('load_ext', 'autoreload')
else:
    get_ipython().run_line_magic('reload_ext', 'autoreload')
%autoreload 2

# --- System and Path --- #
import os
import sys
REPO_PATH = os.path.abspath(os.path.join('..'))
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)
import warnings
warnings.filterwarnings("ignore")

# --- Data Manipulation --- #
import pandas as pd
import numpy as np

In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
2110572 Take-Home Exam: Short Answer Scoring with AutoGluon
Author: Your Name

Usage:
  1) Ensure train.csv, test.csv, and sample_submission.csv are in the same directory.
  2) Install AutoGluon: pip install autogluon.multimodal==0.7.0 (for example)
  3) Run this script as a .py or within a Jupyter notebook.
"""

import os
import pandas as pd
from autogluon.multimodal import MultiModalPredictor

def combine_text(question, answer):
    """Combine question and answer into one string."""
    q = question if pd.notna(question) else ""
    a = answer   if pd.notna(answer)   else ""
    return q + " " + a

def main():
    # -------------------------
    # 1. Read data
    # -------------------------
    train_path = os.path.join(REPO_PATH, "data", "train.csv")
    test_path  = os.path.join(REPO_PATH, "data", "test.csv")
    sample_sub_path = os.path.join(REPO_PATH, "data", "sample_submission.csv")

    if not (os.path.exists(train_path) and
            os.path.exists(test_path)  and
            os.path.exists(sample_sub_path)):
        raise FileNotFoundError("Check that train.csv, test.csv, and sample_submission.csv exist!")

    train_df = pd.read_csv(train_path)
    test_df  = pd.read_csv(test_path)
    sub_df   = pd.read_csv(sample_sub_path)

    # -------------------------
    # 2. Preprocess
    # -------------------------
    # Create a new column "text" by combining question and answer
    train_df["text"] = train_df.apply(lambda row: combine_text(row["question"], row["answer"]), axis=1)
    test_df["text"]  = test_df.apply(lambda row: combine_text(row["question"], row["answer"]), axis=1)

    # The label for regression is the "score"
    # MultiModalPredictor expects a single "label" column for the target.
    # We'll rename "score" -> "label" in the train DataFrame.
    train_df.rename(columns={"score": "label"}, inplace=True)

    # For convenience, drop columns we won't directly need
    # but keep ID so we can join predictions to test data if needed
    # We only truly need 'text' and 'label' columns for training
    keep_cols_train = ["text", "label"]  # plus any extra numeric/categorical features if desired
    train_df = train_df[keep_cols_train]

    # In test data, we do not have "score", so we only keep "ID" and "text"
    # so that we can align predictions properly in submission
    keep_cols_test = ["ID", "text"]
    test_df = test_df[keep_cols_test]

    # -------------------------
    # 3. Initialize Predictor
    # -------------------------
    # We specify problem_type='regression' to predict continuous scores
    predictor = MultiModalPredictor(
        label="label",
        problem_type="regression"
    )

    # -------------------------
    # 4. Fit the Model
    # -------------------------
    # You can use various presets (e.g. "medium_quality_finetune", "high_quality", etc.)
    # and set a time_limit if desired:
    predictor.fit(
        train_data=train_df,
        presets="medium_quality",
        time_limit=60*1  # (in seconds) or set None
    )

    # -------------------------
    # 5. Predict on Test Data
    # -------------------------
    # We only pass the text column at prediction time.
    # Since the predictor expects the same column used at training,
    # we keep a DataFrame with a "text" column but no "label".
    predictions = predictor.predict(test_df)

    # -------------------------
    # 6. Prepare Submission
    # -------------------------
    # The predictions are our "score" values.
    # Insert them into sub_df and save to "submission.csv".
    sub_df["score"] = predictions
    sub_df.to_csv("submission.csv", index=False)
    print("Submission saved to submission.csv")

if __name__ == "__main__":
    main()


No path specified. Models will be saved in: "AutogluonModels/ag-20250307_085204"
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.11.10
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.3.0: Thu Jan  2 20:23:36 PST 2025; root:xnu-11215.81.4~3/RELEASE_ARM64_T8112
CPU Count:          8
Pytorch Version:    2.5.1
CUDA Version:       CUDA is not available
Memory Avail:       3.31 GB / 8.00 GB (41.3%)
Disk Space Avail:   23.03 GB / 228.27 GB (10.1%)

AutoMM starts to create your model. ✨✨✨

To track the learning progress, you can open a terminal and launch Tensorboard:
    ```shell
    # Assume you have installed tensorboard
    tensorboard --logdir /Users/pupipatsingkhorn/Developer/repositories/NLP/nlp-2025-midterm-kaggle-asas/notebooks/AutogluonModels/ag-20250307_085204
    ```

Seed set to 0
GPU Count: 0
GPU Count to be Used: 0

GPU available: True (mps), used: True
TPU available: False, using

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 0, global step 1: 'val_rmse' reached 1.11737 (best 1.11737), saving model to '/Users/pupipatsingkhorn/Developer/repositories/NLP/nlp-2025-midterm-kaggle-asas/notebooks/AutogluonModels/ag-20250307_085204/epoch=0-step=1.ckpt' as top 3


...

Predicting: |          | 0/? [00:00<?, ?it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]

AutoMM has created your model. 🎉🎉🎉

To load the model, use the code below:
    ```python
    from autogluon.multimodal import MultiModalPredictor
    predictor = MultiModalPredictor.load("/Users/pupipatsingkhorn/Developer/repositories/NLP/nlp-2025-midterm-kaggle-asas/notebooks/AutogluonModels/ag-20250307_085204")
    ```

If you are not satisfied with the model, try to increase the training time, 
adjust the hyperparameters (https://auto.gluon.ai/stable/tutorials/multimodal/advanced_topics/customization.html),
or post issues on GitHub (https://github.com/autogluon/autogluon/issues).




Predicting: |          | 0/? [00:00<?, ?it/s]

Submission saved to submission.csv
